In [1]:
# config.py

#--------Packages
from pathlib import Path

### Below from SMM notes
import pandas as pd
import matplotlib.pyplot as plt
import dask
from dask import delayed, compute
from dask.distributed import Client
import scipy.optimize as optimize
from scipy.optimize import minimize
import scipy.stats as stats
import plotly.express as px
import numpy as np
import sys
from functools import lru_cache
import warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize


!{sys.executable} -m pip install quantecon
import quantecon as qe
from quantecon.markov.approximation import tauchen

#---------relative path configuration

# Root = folder containing config.py

from pathlib import Path

try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    ROOT = Path.cwd()

SCRIPTS = ROOT / "scripts"
RESULTS = ROOT / "results"

RESULTS.mkdir(exist_ok=True)

print("Setup successful")
print(f"ROOT: {ROOT}")
print(f"SCRIPTS: {SCRIPTS}")
print(f"RESULTS: {RESULTS}")



Setup successful
ROOT: /Users/marlifirillo/CompEcon_Fall25/DynamicProgramming/ProblemSet8
SCRIPTS: /Users/marlifirillo/CompEcon_Fall25/DynamicProgramming/ProblemSet8/scripts
RESULTS: /Users/marlifirillo/CompEcon_Fall25/DynamicProgramming/ProblemSet8/results


In [2]:
#Module 1: Define the model parameters and set up targets moments from Table 3.

#--------CE Calibrated Parameters

beta = 0.95      # discount factor
delta = 0.15     # depreciation rate
p = 1.0          # price of capital

#--------SMM starting parameters

alpha = 0.6956
gamma = 0.1331
rho = 0.0976
sigma = 0.8932
phi_tilde0 = 0.0


#Psi d - AKA target-moments

target_moments = [
    0.03,   # a1 - coefficient on expected average Q
    0.24,   # a2 - coefficient on the profit rate
    0.40,   # serial correlation of investment rate
    0.25,   # std of profit rate (pi/K)
    3.00,   # average Q
    0.25    # fraction externally financed
]


# Parameter names used in SMM estimation
PARAMETER_NAMES = [
    "alpha",
    "gamma",
    "rho",
    "sigma",
    "phi_tilde0"
]

# Six moments matched between simulated and actual data
MOMENT_NAMES = [
    "a1",
    "a2",
    "sc(I/K)",
    "std(CF/K)",
    "mean Q",
    "external fraction"
]

#target_moments = np.asarray(target_moments, dtype=float)

# weighting_matrix = np.eye(len(target_moments))  # Weighting matrix will be an identity matrix for simplicity



In [3]:
# Module 3: Define functions for use in later steps

# 1. Profit function 
def profit(K, A, alpha):
    """
    Calculate firm profit given capital K and productivity A.
    """
    return A * K**alpha

# 2. Investment function 
def investment(K, K_next, delta):
    """
    Calculate gross investment needed to move from K to K_next.
    """
    return K_next - (1 - delta) * K

# 3. Adjustment costs now use GROSS investment from Equation (4)
def adjustment_costs(K, K_next, delta, gamma):
    """
    Calculate adjustment costs using gross investment from Equation (4).
    """
    I = investment(K, K_next, delta)
    return (gamma / 2) * (I / K) ** 2 * K


# Calculate current period net cash flow
def current_payout(K, A, alpha, K_next, delta, gamma):
    """
    Calculate current period net cash flow after investment and adjustment costs.
    """
    return (
        profit(K, A, alpha)
        - investment(K, K_next, delta)
        - adjustment_costs(K, K_next, delta, gamma)
    )

In [ ]:
# Module 4: Settings for simulation and standard error exercise
# Settings used for simulation and inference
INFERENCE_CONFIG = dict(
    K_min=1.0,
    K_max=1000.0,
    nK=100,
    nA=5,
    num_firms=500,
    num_periods=100,
    burn_in=30,
    delta=float(delta),
    beta=float(beta),
    phi1=0.0
)

# Number of simulated panels used for each SMM objective
S_ESTIMATION = 1

# Number of independent panels used to estimate moment covariance
R_COVARIANCE = 100

# Number of panels used to reduce noise in numerical derivatives
S_DERIVATIVE = 10

# Seeds make simulated results reproducible
ESTIMATION_SEEDS = tuple(
    12345 + s for s in range(S_ESTIMATION)
)

DERIVATIVE_SEEDS = tuple(
    54321 + s for s in range(S_DERIVATIVE)
)

COVARIANCE_SEEDS = tuple(
    90000 + r for r in range(R_COVARIANCE)
)

# Bounds for each parameter during estimation
INFERENCE_BOUNDS = np.array([
    (0.40, 0.95),
    (0.001, 2.0),
    (0.001, 0.98),
    (0.05, 1.5),
    (0.0, 2.0)
])

# Step sizes used to calculate numerical derivatives
FD_STEPS = np.array([
    0.005,
    0.01,
    0.01,
    0.02,
    0.02
])

# Maximum number of times the optimizer can evaluate the objective
MAX_FUNCTION_EVALUATIONS = 1000

#Define grids


def make_A_process(rho, sigma, nA=5):
    # QuantEcon tauchen method wrapper
    mc = tauchen(n=nA, rho=rho, sigma=sigma)
    logA_grid = np.asarray(mc.state_values)
    P = np.asarray(mc.P)

    # Ensure transition probabilities are valid
    P = np.where(P < 0, 0.0, P)
    P = P / P.sum(axis=1, keepdims=True)
    A_grid = np.exp(logA_grid)

    return A_grid, P

def make_K_grid(K_min=1.0, K_max=1000.0, nK=300):
    return np.linspace(K_min, K_max, nK)




In [ ]:
# Module 5: Solve DP model, simulate firms  

def solve_inference_dp(theta, cfg):
    """
    Solve the model for a given set of parameters using value function iteration.

    Returns the grids, transition matrix, value function, capital policy,
    and financing policy used to simulate the model.
    """
    alpha, gamma, rho, sigma, phi_tilde0 = np.asarray(theta, dtype=float)

    # Create productivity process and capital grid
    A_grid, P = make_A_process(rho, sigma, nA=cfg["nA"])
    K_grid = make_K_grid(
        cfg["K_min"],
        cfg["K_max"],
        nK=cfg["nK"]
    )

    # Reshape grids to evaluate all combinations of K, K_next, and A
    K = K_grid[:, None, None]
    K_next = K_grid[None, :, None]
    A = A_grid[None, None, :]

    # Calculate profit and investment for each possible choice
    pi = profit(K, A, alpha)
    I = investment(K, K_next, cfg["delta"])

    # Determine whether external financing is needed
    external = K_next > (
        pi + (1.0 - cfg["delta"]) * K
    )

    # Calculate payout before external financing costs
    net_funds = (
        pi
        - I
        - adjustment_costs(
            K,
            K_next,
            cfg["delta"],
            gamma
        )
    )

    # Apply external financing costs when external financing is needed
    equity_issuance_cost = (
        phi_tilde0
        + cfg["phi1"] * (I - pi)
    )

    payout = net_funds - np.where(
        external,
        equity_issuance_cost,
        0.0
    )

    # Initialize value function
    V = np.zeros((len(K_grid), len(A_grid)))

    # Value function iteration
    for iteration in range(1, 1001):

        # Expected value for each possible choice of K_next
        expected_value = V @ P.T

        # Bellman equation
        rhs = (
            payout
            + cfg["beta"] * expected_value[None, :, :]
        )

        # Find optimal K_next for each K and A
        policy = np.argmax(rhs, axis=1)

        # Store value associated with optimal choice
        V_new = np.take_along_axis(
            rhs,
            policy[:, None, :],
            axis=1
        )[:, 0, :]

        # Check convergence
        V_dist = np.max(np.abs(V_new - V))
        V = V_new

        if V_dist < 1e-6:
            break

    else:
        raise ValueError(
            "Inference DP did not converge."
        )

    # Record whether optimal choice uses external financing
    finance_policy = np.take_along_axis(
        external,
        policy[:, None, :],
        axis=1
    )[:, 0, :].astype(int)

    return K_grid, A_grid, P, V, policy, finance_policy

#################### Part2 ####################

def simulate_panel(theta, solved, seed, cfg):
    """
    Simulate a panel of firms using the solved policy function.

    Generates investment, cash flow, Q, expected Q, and financing data
    for each firm and period after the burn-in period.
    """
    alpha, gamma, rho, sigma, phi_tilde0 = np.asarray(theta, dtype=float)

    K_grid, A_grid, P, V, policy, finance_policy = solved

    num_firms = cfg["num_firms"]
    num_periods = cfg["num_periods"]
    burn_in = cfg["burn_in"]

    if num_firms < 2 or not 0 <= burn_in < num_periods - 1:
        raise ValueError(
            "Need at least two firms and two retained periods."
        )

    # Generate random draws used for productivity transitions
    rng = np.random.default_rng(seed)
    uniforms = rng.random((num_firms, num_periods))

    # Start all firms in the middle of the K and A grids
    K_index = np.full(
        num_firms,
        len(K_grid) // 2,
        dtype=int
    )

    A_index = np.full(
        num_firms,
        len(A_grid) // 2,
        dtype=int
    )

    retained_periods = num_periods - burn_in

    # Store simulated firm data
    investment_rate = np.empty(
        (num_firms, retained_periods)
    )

    cash_flow = np.empty(
        (num_firms, retained_periods)
    )

    expected_Q = np.empty(
        (num_firms, retained_periods)
    )

    Q = np.empty(
        (num_firms, retained_periods)
    )

    financing = np.empty(
        (num_firms, retained_periods)
    )

    # Cumulative probabilities used to draw next-period productivity
    cumulative_P = np.cumsum(P, axis=1)
    cumulative_P[:, -1] = 1.0

    # Simulate firms over time
    for t in range(num_periods):

        # Choose next-period capital using the policy function
        K_next_index = policy[K_index, A_index]

        K = K_grid[K_index]
        K_next = K_grid[K_next_index]
        A = A_grid[A_index]

        # Keep observations after burn-in
        if t >= burn_in:
            j = t - burn_in

            # Investment divided by capital
            investment_rate[:, j] = (
                investment(
                    K,
                    K_next,
                    cfg["delta"]
                ) / K
            )

            # Cash flow divided by capital
            cash_flow[:, j] = (
                profit(K, A, alpha) / K
            )

            # Expected Q used in the investment regression
            expected_Q[:, j] = (
                np.sum(
                    P[A_index, :] * V[K_next_index, :],
                    axis=1
                ) / K_next
            )

            # Current Q
            Q[:, j] = (
                V[K_index, A_index] / K
            )

            # External financing decision
            financing[:, j] = (
                finance_policy[K_index, A_index]
            )

        # Move capital to next period
        K_index = K_next_index

        # Draw next-period productivity
        A_index = np.sum(
            uniforms[:, t, None]
            > cumulative_P[A_index, :],
            axis=1
        )

    return {
        "investment_rate": investment_rate,
        "cash_flow": cash_flow,
        "expected_Q": expected_Q,
        "Q": Q,
        "financing": financing
    }




In [7]:
#Module 6:

def calculate_moments(panel):
    """
    Calculate the six moments used for SMM from the simulated panel.

    Uses the simulated investment, cash flow, Q, and financing data
    to calculate the moments matched to the actual data.
    """
    investment_rate = panel["investment_rate"]
    cash_flow = panel["cash_flow"]
    expected_Q = panel["expected_Q"]
    Q = panel["Q"]
    financing = panel["financing"]

    # Regression of investment on expected Q and cash flow
    X = np.column_stack((
        np.ones(investment_rate.size),
        expected_Q.ravel(),
        cash_flow.ravel()
    ))

    if np.linalg.matrix_rank(X) < 3:
        raise ValueError(
            "Simulated investment regression is rank deficient."
        )

    regression_results = np.linalg.lstsq(
        X,
        investment_rate.ravel(),
        rcond=None
    )[0]

    _, a1, a2 = regression_results

    # Calculate serial correlation of investment
    investment_lag = investment_rate[:, :-1].ravel()
    investment_lead = investment_rate[:, 1:].ravel()

    if min(
        np.std(investment_lag),
        np.std(investment_lead)
    ) < 1e-12:
        raise ValueError(
            "Investment serial correlation is undefined at this parameter trial."
        )

    # Calculate six simulated moments
    moments = np.array([
        a1,
        a2,
        np.corrcoef(
            investment_lag,
            investment_lead
        )[0, 1],
        np.std(cash_flow),
        np.mean(Q),
        np.mean(financing)
    ])

    if not np.isfinite(moments).all():
        raise ValueError(
            "Non-finite simulated moments."
        )

    return moments


def inference_panel_moments(theta, solved, seed, cfg):
    """
    Simulate one panel and calculate its six moments.
    """
    panel = simulate_panel(
        theta,
        solved,
        seed,
        cfg
    )

    return calculate_moments(panel)


def inference_mean_moments(theta, seeds, cfg=INFERENCE_CONFIG):
    """
    Calculate average simulated moments across panels using different seeds.

    The model is solved once for each parameter set and the resulting
    policy function is used for each simulated panel.
    """
    solved = solve_inference_dp(theta, cfg)

    panel_moments = [
        inference_panel_moments(
            theta,
            solved,
            seed,
            cfg
        )
        for seed in seeds
    ]

    return np.mean(panel_moments, axis=0)

In [8]:
def estimate_inference_stage(start, W, label, maxfev=MAX_FUNCTION_EVALUATIONS):
    """Powell exploration followed by a local simplex refinement of its best trial."""
    W = np.asarray(W, dtype=float)
    if W.shape != (6, 6) or not np.isfinite(W).all():
        raise ValueError("W must be a finite 6-by-6 matrix.")
    history, cache = [], {}
    best = {"theta": np.array(start, dtype=float), "loss": np.inf}
    def raw_objective(theta):
        key = tuple(np.asarray(theta, dtype=float))
        if key in cache:
            return cache[key]
        try:
            m = inference_mean_moments(theta, ESTIMATION_SEEDS)
            gap = m - np.asarray(target_moments)
            loss = float(gap @ W @ gap)
        except ValueError:
            loss = 1e12
        cache[key] = loss
        if loss < best["loss"]:
            best.update(theta=np.array(theta).copy(), loss=loss)
        history.append(loss)
        if len(history) == 1 or len(history) % 25 == 0:
            print(f"{label}: evaluation {len(history)}, best loss {best['loss']:.6f}", flush=True)
        return loss
    scale = max(raw_objective(start), 1e-8)
    objective = lambda theta: raw_objective(theta) / scale
    exploration = minimize(objective, np.asarray(start, dtype=float), method="Powell",
                           bounds=INFERENCE_BOUNDS,
                           options=dict(maxfev=maxfev, xtol=1e-3, ftol=1e-3))
    # Grid-induced discontinuities can cause Powell to return a worse terminal point.
    # Refine the best evaluated point and explicitly check the terminal result.
    simplex = np.tile(best["theta"], (6, 1))
    for j in range(5):
        h = FD_STEPS[j]
        direction = 1 if best["theta"][j] + h <= INFERENCE_BOUNDS[j, 1] else -1
        simplex[j+1, j] += direction * h
    fit = minimize(objective, best["theta"], method="Nelder-Mead", bounds=INFERENCE_BOUNDS,
                   options=dict(maxfev=maxfev, initial_simplex=simplex,
                                xatol=1e-4, fatol=1e-6))
    terminal_loss = raw_objective(fit.x)
    terminal_is_best = abs(terminal_loss - best["loss"]) <= 1e-6 * max(1.0, best["loss"])
    converged = bool(fit.success and terminal_is_best)
    print(f"{label}: converged={converged}; {fit.message}", flush=True)
    if best["loss"] >= 1e12:
        raise ValueError("No valid parameter trial found.")
    return dict(theta=best["theta"], loss=best["loss"], optimizer=fit,
                exploration=exploration, converged=converged, history=history)


def estimate_moment_covariance(theta, seeds=COVARIANCE_SEEDS, cfg=INFERENCE_CONFIG):
    """Covariance of ONE panel's six moments; R is not S in the SE correction."""
    if len(seeds) <= 6 or len(set(seeds)) != len(seeds):
        raise ValueError("Use more than six distinct independent panel seeds.")
    solved = solve_inference_dp(theta, cfg)
    rows = []
    for i, seed in enumerate(seeds, start=1):
        rows.append(inference_panel_moments(theta, solved, seed, cfg))
        if i % 25 == 0:
            print(f"Covariance panels: {i}/{len(seeds)}", flush=True)
    rows = np.asarray(rows)
    Omega = np.cov(rows, rowvar=False, ddof=1)
    # A pseudoinverse can manufacture misleading precision; require invertibility.
    eigenvalues = np.linalg.eigvalsh(Omega)
    if eigenvalues[0] <= 0 or np.linalg.cond(Omega) > 1e12:
        raise ValueError("Moment covariance is singular or ill conditioned; increase panels or diagnose redundant moments.")
    W_star = np.linalg.solve(Omega, np.eye(6))
    return rows, Omega, (W_star + W_star.T) / 2

In [9]:
def numerical_moment_jacobian(theta, steps=FD_STEPS, seeds=DERIVATIVE_SEEDS):
    """6-by-5 finite difference Jacobian, with common draws and one-sided boundary differences."""
    theta = np.asarray(theta, dtype=float)
    base = inference_mean_moments(theta, seeds)
    D = np.empty((6, 5))
    methods = []
    for j, h in enumerate(steps):
        lower, upper = INFERENCE_BOUNDS[j]
        plus, minus = theta.copy(), theta.copy()
        plus[j] += h
        minus[j] -= h
        if minus[j] >= lower and plus[j] <= upper:
            D[:, j] = (inference_mean_moments(plus, seeds) - inference_mean_moments(minus, seeds)) / (2*h)
            methods.append("central")
        elif plus[j] <= upper:
            D[:, j] = (inference_mean_moments(plus, seeds) - base) / h
            methods.append("forward")
        elif minus[j] >= lower:
            D[:, j] = (base - inference_mean_moments(minus, seeds)) / h
            methods.append("backward")
        else:
            raise ValueError("Finite-difference step is larger than the allowed parameter interval.")
    return D, methods


def standard_errors_from_jacobian(D, W_star, S=S_ESTIMATION):
    """Assignment formula (1+1/S)*(D' W* D)^(-1), with no extra sample-size division."""
    if D.shape != (6, 5) or S < 1:
        raise ValueError("Need a 6-by-5 Jacobian and S >= 1.")
    # With Omega equal to the covariance of panel-level moments, it already
    # contains the finite-panel sampling scale. Do not divide by firms, T, or R again.
    whitened = np.linalg.cholesky(W_star).T @ D
    if np.linalg.matrix_rank(whitened) < 5:
        raise ValueError("Moment Jacobian has rank below 5: ordinary standard errors are not identified on this grid.")
    information = D.T @ W_star @ D
    information = (information + information.T) / 2
    if np.linalg.cond(information) > 1e12:
        raise ValueError("D' W D is too ill conditioned for reliable standard errors.")
    covariance_theta = (1 + 1 / S) * np.linalg.solve(information, np.eye(5))
    covariance_theta = (covariance_theta + covariance_theta.T) / 2
    if np.any(np.diag(covariance_theta) <= 0):
        raise ValueError("Parameter covariance has nonpositive diagonal entries.")
    return covariance_theta, np.sqrt(np.diag(covariance_theta))

In [10]:
theta_start_inference = np.array([alpha, gamma, rho, sigma, phi_tilde0], dtype=float)
stage1 = estimate_inference_stage(theta_start_inference, np.eye(6), "Identity stage")
theta_stage1 = stage1["theta"]
print("Stage-1 estimates:", dict(zip(PARAMETER_NAMES, theta_stage1)))
print("Stage-1 converged:", stage1["converged"])

Identity stage: evaluation 1, best loss 0.331009
Identity stage: evaluation 25, best loss 0.276967
Identity stage: evaluation 50, best loss 0.188358
Identity stage: evaluation 75, best loss 0.185588
Identity stage: evaluation 100, best loss 0.123693
Identity stage: evaluation 125, best loss 0.112608
Identity stage: evaluation 150, best loss 0.092775
Identity stage: evaluation 175, best loss 0.068014
Identity stage: evaluation 200, best loss 0.067496
Identity stage: evaluation 225, best loss 0.067496
Identity stage: evaluation 250, best loss 0.067496
Identity stage: evaluation 275, best loss 0.067496
Identity stage: evaluation 300, best loss 0.067448
Identity stage: evaluation 325, best loss 0.067001
Identity stage: evaluation 350, best loss 0.066952
Identity stage: evaluation 375, best loss 0.066901
Identity stage: evaluation 400, best loss 0.066843
Identity stage: evaluation 425, best loss 0.066782
Identity stage: evaluation 450, best loss 0.066779
Identity stage: evaluation 475, best

In [11]:
#Construct W matrix for stage 2

replicated_moments, moment_covariance, W_star = estimate_moment_covariance(
    theta_stage1
)

stage2 = estimate_inference_stage(theta_stage1, W_star, "Efficient stage")
theta_stage2 = stage2["theta"]
print("Stage-2 estimates:", dict(zip(PARAMETER_NAMES, theta_stage2)))
print("Stage-2 converged:", stage2["converged"])
moments_stage2 = inference_mean_moments(theta_stage2, ESTIMATION_SEEDS)
print(pd.DataFrame({"Target": target_moments, "Stage 2": moments_stage2}, index=MOMENT_NAMES).to_string(float_format=lambda x: f"{x:.4f}"))

Covariance panels: 25/100
Covariance panels: 50/100
Covariance panels: 75/100
Covariance panels: 100/100
Efficient stage: evaluation 1, best loss 17906.567856
Efficient stage: evaluation 25, best loss 17906.567856
Efficient stage: evaluation 50, best loss 17906.567856
Efficient stage: evaluation 75, best loss 17906.567856
Efficient stage: evaluation 100, best loss 17906.567856
Efficient stage: evaluation 125, best loss 17906.567856
Efficient stage: evaluation 150, best loss 17823.406508
Efficient stage: evaluation 175, best loss 17809.724939
Efficient stage: evaluation 200, best loss 17796.490036
Efficient stage: evaluation 225, best loss 17790.774050
Efficient stage: evaluation 250, best loss 17789.912755
Efficient stage: evaluation 275, best loss 17788.750237
Efficient stage: evaluation 300, best loss 17785.647592
Efficient stage: evaluation 325, best loss 17762.578691
Efficient stage: evaluation 350, best loss 17761.405699
Efficient stage: evaluation 375, best loss 17761.350177
Effi

In [12]:
jacobian, derivative_methods = numerical_moment_jacobian(theta_stage2, FD_STEPS)
jacobian_half, derivative_methods_half = numerical_moment_jacobian(theta_stage2, FD_STEPS / 2)
parameter_covariance, standard_errors = standard_errors_from_jacobian(jacobian, W_star, S_ESTIMATION)
parameter_covariance_half, standard_errors_half = standard_errors_from_jacobian(jacobian_half, W_star, S_ESTIMATION)
relative_derivative_change = (np.linalg.norm(jacobian - jacobian_half, axis=0)
                              / np.maximum(np.linalg.norm(jacobian_half, axis=0), 1e-12))
near_bound = np.minimum(theta_stage2 - INFERENCE_BOUNDS[:, 0],
                        INFERENCE_BOUNDS[:, 1] - theta_stage2) < 1e-4
stable_derivatives = bool(np.all(relative_derivative_change < 0.25))
inference_checks_pass = bool(stage1["converged"] and stage2["converged"]
                             and stable_derivatives and not np.any(near_bound))
standard_errors_table = pd.DataFrame({
    "Identity estimate": theta_stage1,
    "Efficient estimate": theta_stage2,
    "SE (h)": standard_errors,
    "SE (h/2)": standard_errors_half,
    "Derivative change": relative_derivative_change,
    "Difference method": derivative_methods,
    "At bound": near_bound,
}, index=PARAMETER_NAMES)
print(standard_errors_table.to_string(float_format=lambda x: f"{x:.4f}"))
print("Status:", "Numerical checks passed" if inference_checks_pass else "PROVISIONAL: review convergence, derivative sensitivity and boundary diagnostics")
print(f"S={S_ESTIMATION}; R={R_COVARIANCE}; simulation adjustment={1 + 1/S_ESTIMATION:.4f}")
print("Jacobian shape:", jacobian.shape)
print("Jacobian rank:", np.linalg.matrix_rank(jacobian))
print("Information matrix condition number:", np.linalg.cond(jacobian.T @ W_star @ jacobian))

            Identity estimate  Efficient estimate  SE (h)  SE (h/2)  Derivative change Difference method  At bound
alpha                  0.7021              0.7020  0.0032    0.0003             0.9140           central     False
gamma                  0.9690              0.9691  0.0042    0.0009             0.5104           central     False
rho                    0.4065              0.4069  0.0088    0.0065             0.5468           central     False
sigma                  0.4468              0.4476  0.0021    0.0016             0.6644           central     False
phi_tilde0             1.2352              1.2321  0.0133    0.0121             0.4970           central     False
Status: PROVISIONAL: review convergence, derivative sensitivity and boundary diagnostics
S=1; R=100; simulation adjustment=2.0000
Jacobian shape: (6, 5)
Jacobian rank: 5
Information matrix condition number: 876.9173994042036
